# 7. 지시를 따르도록 fine-tuning
* 챗봇 애플리케이션, 개인 비서, 그 외 다른 대화식 작업을 위한 LLM을 개발하는데 필요

## 7.1. instruction fine-tuning (supervised instruction fine-tuning)

* 사전 훈련된 LLM은 구체적인 명령은 잘 수행하지 못함
* instruction fine-tuning에서 핵심은 dataset 준비

## 7.2. supervised instruction fine-tuning을 위해 dataset 준비

#### 데이터셋 다운로드
* 지시-응답 쌍 1100개 (다른 공개 데이터셋은 Appendix B)

In [23]:
import json
import os
import urllib


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r") as file:
        data = json.load(file)
    return data


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("샘플 개수:", len(data))

샘플 개수: 1100


In [10]:
print("샘플 예시:\n", data[50])

샘플 예시:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


#### input이 비어 있는 경우도 있음

In [11]:
print("다른 샘플:\n", data[999])

다른 샘플:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


#### 지시 미세 튜닝 훈련 방식
* 입력-출력 쌍으로 구성된 데이터셋에서 모델 훈련

* 샘플 포맷팅 방법 - 프롬프트 스타일 (prompt style)
    - Alpaca, Phi-3을 훈련하는 데 사용됨
    - Alpaca 스타일이 초기 fine-tuning 방법을 정의하는 데 크게 기여 + 인기 많은 포맷

#### data 리스트의 항목을 Alpaca 스타일의 포맷으로 변환

* input 필드가 비어 있는 경우 ### Input: 섹션을 건너뜀

In [12]:
# 딕셔너리 entry를 입력으로 받아 포맷팅된 문자열 만듦
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )

    return instruction_text + input_text

In [13]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


In [14]:
# input 필드가 없는 샘플은 결과에 ### Input: 섹션이 포함되지 않음
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


#### 데이터셋 분할 (train: 0.85, validation: 0.1, test: 0.05)

In [15]:
train_portion = int(len(data) * 0.85)  # 훈련에 데이터의 85% 사용
test_portion = int(len(data) * 0.1)  # 테스트에 10% 사용 
val_portion = len(data) - train_portion - test_portion  # 검증에 남은 5% 사용

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("훈련 세트 크기:", len(train_data))
print("검증 세트 크기:", len(val_data))
print("테스트 세트 크기:", len(test_data))

훈련 세트 크기: 935
검증 세트 크기: 55
테스트 세트 크기: 110


## 7.3. 훈련 배치 만들기

#### 이전 chapter DataLoader 클래스 -> 훈련 배치를 자동으로 만듦
* 이 클래스는 샘플 리스트를 배치로 묶어주는 기본 콜레이트(collate) 함수를 사용
    - collate 함수: 훈련하는 동안 개별 데이터 샘플의 리스트를 받아 하나의 배치로 합쳐 모델이 효과적으로 처리하도록 함

#### instruction fine-tuning
* 구성이 더 복잡해서 Dataloader에 적용할 사용자 정의 콜레이트 함수 정의 필요

#### 배치 처리 과정
* 2.1, 2.2) InstructionDataset 클래스: 모든 샘플에 format_input 적용 + 토큰화 (SpamDataset 클래스와 비슷)
    - InstructionDataset 클래스의 생성자인 __init__ 메서드 안에 구현

####  지시 데이터셋 클래스 (InstructionDataset)

In [ ]:
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:  # 텍스트 토큰화
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

#### <|endoftext|> 토큰 사용
* 분류 fine-tuning처럼 여러 개 훈련 샘플을 배치로 묶어서 훈련 속도를 높여야 하고, 이를 위해 입력 길이를 맞추는 패딩 추가 필요
* <|endoftext|> 토큰의 ID를 토큰화된 입력에 바로 추가

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

#### dataloader에 전달한 사용자 정의 콜레이트 함수
* 배치에 있는 훈련 샘플의 길이를 맞추기 위해 패딩 추가
* 아래 그림의 방식: 전체 데이터셋이 아니라 각 배치에서 긴 샘플에 맞춰 시퀀스를 확장 (불필요한 패딩 최소화)


In [ ]:
import torch

def custom_collate_draft_1(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)  # 배치에서 가장 긴 시퀀스 찾기
    inputs_lst = []

    for item in batch:  # 입력에 패딩 추가
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )

        inputs = torch.tensor(padded[:-1])  # 이전에 추가로 넣은 패딩 토큰 제외
        inputs_lst.append(inputs)

    inputs_tensor = torch.stack(inputs_lst).to(device)  # 입력의 리스트를 텐서로 변환하고 타깃 장치로 전송합니다.
    return inputs_tensor

In [ ]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

#### 입력 토큰 ID에 해당하는 타깃 토큰 ID로 구성된 배치도 만들어야 함
* 타깃 ID: 모델이 생성해야 할 것
* 훈련 과정에서 손실을 계산하여 가중치 업데이트에 중요함
* 사용자 정의 콜레이트 함수 수정: 입력 토큰 ID와 타겟 토큰 ID를 반환하도록 수정
    - LLM 사전훈련과 비슷하게, 타겟 토큰 ID는 입력 토큰 ID와 같지만 오른쪽으로 한 위치씩 이동한 것
    - 덕분에 LLM이 시퀀스에 있는 다음 토큰을 예측하는 방법을 학습할 수 있음
* 입력 토큰 ID에서 타겟 토큰 ID를 생성하는 콜레이트 함수

In [ ]:
def custom_collate_draft_2(
    batch,
    pad_token_id=50256,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )

        inputs = torch.tensor(padded[:-1])  # 입력에서 마지막 토큰 삭제
        targets = torch.tensor(padded[1:])  # 오른쪽으로 하나 이동하여 타겟 만들기
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor


inputs, targets = custom_collate_draft_2(batch)
print("입력: ", inputs)
print("타겟:", targets)

#### -100으로 바꾸기
* 모든 패딩 토큰을 placeholder 값 -100으로 바꿈 
    - -100: 훈련 손실 계산에서 패딩 토큰 제외
    - allowed_max_length 매개변수를 추가하여 선택적으로 샘플 길이를 제한하도록 만듦
    (데이터셋에 GPT-2 모델이 지원하는 1,024 토큰의 문맥 길이보다 긴 샘플이 있을 때 유용)
* ID가 50256인 텍스트 종료 토큰 하나를 남겨둠
    - LLM이 지시에 응답할 때 텍스트 종료 토큰을 언제 생성할지 학습할 수 있음
    - 이 토큰은 응답 생성이 완료되었다는 걸 나타냄

In [ ]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]

        # max_length까지 시퀀스 패딩
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )

        inputs = torch.tensor(padded[:-1])  # 입력에서 마지막 토큰 삭제
        targets = torch.tensor(padded[1:])  # 오른쪽으로 한 칸 이동하여 타겟 생성

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            
            # 타겟 첫 번째를 제외한 모든 패딩 토큰을 ignore_index로 변환
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            # 최대 시퀀스 길이로 절단
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [ ]:
inputs, targets = custom_collate_fn(batch)
print("입력: ", inputs)
print("\n타겟:", targets)

#### 위 작업의 논리
* 간단한 샘플
    - 샘플에 있는 각각의 출력 로짓은 모델의 어휘사전에 있는 가증한 토큰에 해당
* 훈련 과정에서 모델이 토큰의 시퀀스를 에측할 때 cross-entropy loss를 계산하는 방법은 아래 코드
    - 모델을 사전 훈련하고 분류를 위해 fine-tuning 때 했던 것과 비슷함

In [ ]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],   # 첫 번째 토큰에 대한 예측
     [-0.5, 1.5]]   # 두 번째 토큰에 대한 예측
)

targets_1 = torch.tensor([0,1]) # 정답 토큰 인덱스
loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)

print(loss_1)

#### 토큰 ID를 추가하면 손실 계산에 영향을 미침

In [ ]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # 세 번째 토큰 예측
)

targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

#### 흥미로운 점
* 세 번째 훈련 샘플에 대한 loss: 앞서 2개의 훈련 샘플에서 계산한 손실과 동일
* 즉 cross_entropy loss에서 target_3 벡터에 있는 세 번째 항목인 -100에 해당하는 토큰 무시
    - pytorch의 cross_entropy 기본 설정은 cross_entropy(..., ignore_index=-100)임
    - ignore_index를 사용해 배치에 있는 훈련 샘플의 길이를 동일하게 만들기 위해 추가했던 텍스트 종료 (패딩) 토큰 무시 가능
    - LLM이 응답의 끝을 나타내는 텍스트 종료 토큰을 생성하는 방법을 학습하도록 타겟에서 하나의 50256 토큰을 남겨둬야 함
* 패딩 토큰 마스킹 외에도 지시에 해당하는 타겟 토큰 ID를 마스킹하는 것이 일반적임
    - 지시에 해당하는 타겟 토큰 ID를 마스킹 -> 생성된 응답 타겟 ID에서만 cross_entropy loss를 계산
    - 모델이 지시를 암기하지 않고 정확한 응답을 생성하는데 초점을 맞춰 훈련됨 (과대적합을 줄이는 데 도움)

* 지시 토큰을 마스킹하는 것이 지시 fine-tuning에 유용한지에 대한 갑론을박이 있음
    - "Instruction Tuning With Loss Over Instructions"이라는 논문에서는 지시를 마스킹하지 않는 것이 LLM 성능에 도움이 된다고 밝힘

In [ ]:
targets_3 = torch.tensor([0, 1, -100])
loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

## 7.4. 지시 데이터셋을 위한 데이터 로더
* 지시 fine-tuning을 위해 자동으로 데이터를 섞고 배치 만들기


#### custom_collate_fn 함수의 device 설정
* custom_collate_fn 함수: 입력과 타겟 텐서를 특정 장치로 이동하는 코드 포함
    - torch.stack(input_1st).to(device)
    - 이 장치는 cpu, cuda(NVIDIA GPU가 있으면), mps(애플 실리콘 칩 맥)가 될 수 있음

* 이전에는 메인 훈련 루프에서 데이터를 타겟 장치로 옮겼음
    - 이를 collate 함수에 포함시키면 훈련 루프 밖에서 백그라운드 프로세스로 장치 전송 작업 수행 가능
    - 덕분에 모델 훈련 과정에서 데이터 전송을 기다리느라 GPU가 block 되는 걸 방지할 수 있음

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 애플 실리콘 칩의 GPU를 사용하려면 아래 코드 유지
if torch.backends.mps.is_available():
    device = torch.device("mps")

print("장치:", device)

#### 선택된 장치 설정을 pytorch DataLoader 클래스로 전달되는 custom_collate_fn 함수에서 사용하기
* 파이썬 표준 라이브러리인 functools의 partial 함수 사용
    - 장치 매개변수가 채워진 새로운 custom_collate_fn 함수 만들기
    - GPT-2 모델이 지원하는 최대 문맥 길이로 데이터를 자르기 위해 allowed_max_length=1024 설정

In [ ]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

#### 데이터로더 초기화

In [16]:
from torch.utils.data import DataLoader

num_workers = 0  # 운영체제에서 파이썬 병렬 프로세스를 지원한다면 이 숫자를 증가시킬 수 있음
batch_size = 8

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

NameError: name 'torch' is not defined

#### 각 배치에 있는 각 훈련 샘플의 토큰 개수가 다른 것을 볼 수 있음
* 사용자 정의 콜레이트 함수 -> 데이터 로더가 다른 길이의 배치를 만들 수 있음

In [ ]:
print("훈련 데이터 로더:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

## 7.5. 사전 훈련된 LLM 로드

* 지시 fine-tuning을 시작하기 전에 사전훈련된 GPT 모델 로드
    - 124M 말고 355M로 로드하기
    - 작은 모델은 높은 수준의 지시 수행 작업에 필요한 복잡한 패턴과 미묘한 동작을 학습, 유지하는 데 필요한 용량이 부족함


In [ ]:
import torch
import torch.nn as nn
from torch.nn import LayerNorm

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"])
        )
    def forward(self, x):
        return self.layers(x)

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        # 원하는 출력 차원에 맞도록 투영 차원을 낮춤
        self.head_dim = d_out // num_heads

        # Query, Key, Value 선형 변환
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 여러 head의 출력을 다시 섞어주는 출력 projection
        # Linear층을 사용해 헤드의 출력을 결합
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # causal mask
        # 위쪽 삼각행렬을 1로 만들어서 미래 토큰을 보지 못하게 함
        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        # 텐서 크기: (b, num_tokens, d_out)
        # x: (batch_size, num_tokens, d_in)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # num_heads 차원을 추가함으로써 암묵적으로 행렬 분할
        # 그런 다음 마지막 차원을 num_heads에 맞춰 채움
        # keys, queries, values:
        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)


        # head 차원을 앞으로 이동
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # attention score 계산
        # queries: (b, num_heads, num_tokens, head_dim)
        # keys.transpose(2, 3): (b, num_heads, head_dim, num_tokens)
        # 결과: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # 현재 토큰 수에 맞게 mask 자르기
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 미래 토큰 위치를 -inf로 채움
        # 마스크를 사용해 어텐션 점수를 채움
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # scaled dot-product attention
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # attention weight와 value 곱하기
        # attn_weights: (b, num_heads, num_tokens, num_tokens)
        # values:       (b, num_heads, num_tokens, head_dim)
        # 결과:          (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ values

        # 다시 head 차원을 뒤로 보냄
        # 텐서 크기: (b, num_heads, num_tokens, head_dim) -> (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # 여러 head를 하나로 결합
        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(
            b, num_tokens, self.d_out
        )

        # 마지막 출력 projection (선형 투영)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x # 어텐션 블록을 위한 숏컷 연결
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        shortcut = x # 피드포워드 신경망을 위한 숏컷 연결
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # 원본 입력을 더함

        return x

In [ ]:
import torch.nn as nn

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        
        # 장치 설정을 통해 입력 데이터가 어디에 있는지에 따라 모델을 CPU나 GPU에서 훈련 가능
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"크기가 다릅니다. left: {left.shape}, right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

In [ ]:
import numpy as np

def load_weights_into_gpt(gpt, params): # 위치 임베딩과 토큰 임베딩의 가중치를 params의 값으로 설정
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])): # 모델의 트랜스포커 블록 순회
        q_w, k_w, v_w = np.split( # np.split 함수로 어텐션 가중치와 편향 가중치를 쿼리, 키, 값 세 부분으로 나눔
            params["blocks"][b]["attn"]["c_attn"]["w"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T
        )
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T
        )
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T
        )

        q_b, k_b, v_b = np.split(
            params["blocks"][b]["attn"]["c_attn"]["b"], 3, axis=-1
        )
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b
        )
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b
        )
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b
        )

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"]
        )
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T
        )
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"]
        )

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale,
            params["blocks"][b]["ln_1"]["g"]
        )
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift,
            params["blocks"][b]["ln_1"]["b"]
        )
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale,
            params["blocks"][b]["ln_2"]["g"]
        )
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift,
            params["blocks"][b]["ln_2"]["b"]
        )

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    # 오픈 AI의 원본 GPT-2 모델은 토큰 임베딩의 가중치를 출력 층에 재사용하여 전체 파라미터 개수를 절감하는 가중치 묶기를 사용함
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

In [ ]:
from gpt_download import download_and_load_gpt2

BASE_CONFIG = {
    "vocab_size": 50257,      # 어휘사전 크기
    "context_length": 1024,   # 문맥 길이
    "drop_rate": 0.0,         # 드롭아웃 비율
    "qkv_bias": True          # 쿼리-키-값 편향
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

#### 모델이 가진 기본적인 지시 수행 작업 능력

In [ ]:
torch.manual_seed(123)
input_text = format_input(val_data[0])
print(input_text)

#### generate 함수
* 입력과 출력(모델의 응답)을 결합한 결과 반환
* 사전훈련 LLM과 달리 특정 작업에 대한 모델 성능을 평가할 때는 모델이 생성한 응답에만 초점을 맞추는 경우가 많음

In [19]:
def generate(model, idx, max_new_tokens, context_size,
             temperature=0.0, top_k=None, eos_id=None):

    # for 루프는 이전과 동일. 로짓을 받아 마지막 타임 스텝만 사용
    for _ in range(max_new_tokens):

        # 현재 context_size만큼의 토큰만 모델 입력으로 사용
        idx_cond = idx[:, -context_size:]

        with torch.no_grad():
            logits = model(idx_cond)

        # 마지막 타임스텝의 logits만 사용
        logits = logits[:, -1, :]

        # top-k 샘플링으로 로짓 필터링
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]

            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf")).to(logits.device),
                logits
            )

        # temperature sampling 적용
        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # temperature가 0이면 greedy decoding
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # EOS 토큰을 만나면 생성 중단
        if idx_next == eos_id:
            break

        # 새 토큰을 기존 idx 뒤에 붙이기
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

In [20]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # unsqueeze(0)으로 배치 차원을 추가
    return encoded_tensor

In [21]:
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # 배치 차원 제거
    return tokenizer.decode(flat.tolist())

In [22]:
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256
)

generated_text = token_ids_to_text(token_ids, tokenizer)

NameError: name 'tokenizer' is not defined

#### 모델의 응답 텍스트만 분리하기 위해 generated_text의 시작에서 입력 지시문의 길이를 건너뜀

* 출력 결과
    - 모델이 주어진 지시를 올바르게 수행하는 능력이 아직 없음

In [ ]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

## 7.6. 지시 데이터에서 LLM fine-tuning

In [ ]:
def calc_loss_batch(input_batch, target_batch, model, device):
    # 데이터를 지정된 장치 (GPU)로 전송
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)  # num_batches가 지정되지 않으면 모든 배치 순회
    else:   # num_batches가 데이터 로더에 있는 배치 개수보다 크면 배치 횟수를 데이터 로더에 있는 총 배치 개수로 맞춤
        num_batches = min(num_batches, len(data_loader))
    
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()   # 각 배치의 손실을 더함
        else:
            break
    
    return total_loss / num_batches  # 모든 배치의 손실 평균

In [ ]:
def generate_text_simple(model, idx, # idx는 현재 문맥이 담긴 [batch_size, num_tokens] 크기의 인덱스 배열
                         max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        
        logits = logits[:, -1, :] # 마지막 타임 스텝만 사용하므로 [batch_size, vocab_size]가 [batch_size, num_tokens, vocab_size]에서 [batch_size, vocab_size]로 축소됨
        probas = torch.softmax(logits, dim=-1) # probas는 [batch_size, vocab_size] 크기의 확률 분포
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # idx_next는 [batch_size, 1] 크기의 텐서로, 각 샘플에 대해 가장 높은 확률을 가진 다음 토큰의 인덱스
        idx = torch.cat((idx, idx_next), dim=1) # 선택한 인덱스를 현재 시퀀스에 추가하므로 idx의 크기는 [batch_size, num_tokens + 1]이 됨
    
    return idx

In [ ]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_emb.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " ")) # 간결한 출력 포맷
    model.train()

In [ ]:
def train_model_simple(model, train_loader, val_loader,
                      optimizer, device, num_epochs,
                      eval_freq, eval_iter, start_context, tokenizer):
    
    # 손실과 지금까지 처리한 토큰 수를 추적하기 위해 리스트 초기화
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    
    for epoch in range(num_epochs): # 메인 훈련 루프 시작
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad() # 이전 배치 반복에서 얻은 손실 그래디언트를 초기화
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            loss.backward() # 손실 그래디언트 계산
            optimizer.step() # 손실 그래디언트를 사용하여 모델의 가중치 업데이트
            tokens_seen += input_batch.numel()
            global_step += 1
            
            if global_step % eval_freq == 0:  # 추가적인 평가 단계
                train_loss, val_loss = evaluate_model(
                    model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Epoch {epoch+1} (Step {global_step:06d}): "
                      f"훈련 손실 {train_loss:.3f}, "
                      f"검증 손실 {val_loss:.3f}")
        
        generate_and_print_sample( # 각 에포크 후에 샘플 텍스트 출력
            model, tokenizer, device, start_context
        )
    
    return train_losses, val_losses, track_tokens_seen

In [ ]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval() # 안정적이고 재현 가능한 결과를 위해 평가하는 동안 드롭아웃 비활성화
    with torch.no_grad(): # 계산 오버헤드를 줄이기 위해 훈련 과정에서 필요하지 않은 그래디언트 추적을 끔
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train() # 훈련 모드로 다시 전환
    return train_loss, val_loss

In [ ]:
model.to(device)
torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(
        train_loader, model, device, num_batches=5
    )

    val_loss = calc_loss_loader(
        val_loader, model, device, num_batches=5
    )

print("훈련 손실:", train_loss)
print("검증 손실:", val_loss)

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

In [ ]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA GPU가 잡히지 않았습니다.")

In [ ]:
import gc

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

#### 옵티마이저 초기화, 에포크 횟수 설정, 평가 주기 (훈련 과정 설정)
* 첫 번째 검증 세트 샘플 (val_data[0])을 시작 문맥으로 지정

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1
)

num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분")

#### MPS 캐시 정리

#### 학습 과정에서의 train loss, validation loss
* plot_losses (사전훈련 시 사용)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_losses(epochs_seen, tokens_seen, train_losses, val_losses):
    fig, ax1 = plt.subplots(figsize=(5, 3))

    ax1.plot(epochs_seen, train_losses, label="Training loss")
    ax1.plot(epochs_seen, val_losses, linestyle="--", label="Validation loss")
    ax1.set_xlabel("Epochs")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper right")
    ax1.xaxis.set_major_locator(MaxNLocator(integer=True))

    ax2 = ax1.twiny()
    ax2.plot(tokens_seen, train_losses, alpha=0)
    ax2.set_xlabel("Tokens seen")

    fig.tight_layout()
    plt.show()

In [ ]:
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

#### 손실 그래프: 모델이 효과적으로 훈련 중임
* 응답 품질과 정확성 측면에서는?
    - 응답 추출 + 품질 평가 + 정량화가 가능한 포맷으로 저장할 필요가 있음

## 7.7. 응답 추출 및 저장

#### generate 함수 -> 응답 추출 스텝 수행
* testset에 있는 첫 3개 항목에 대해 모델 응답과 기대되는 응답을 비교하기 쉽도록 함께 출력 

In [ ]:
torch.manual_seed(123)

for entry in test_data[:3]:  # 처음 3개의 테스트 세트 샘플 수집
    input_text = format_input(entry)

    token_ids = generate(  # 7.5절에서 임포트한 generate 함수 사용
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )

    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    print(input_text)
    print(f"\n올바른 응답:\n>> {entry['output']}")
    print(f"\n모델 응답:\n>> {response_text.strip()}")
    print("-----------------------------------------------")

#### 지시 fine-tuning에서는 모델 평가가 어려움
* 다양한 방법
    - 선다형 질문 답변, 사람 평가 (서로 다른 사람의 선호도 비교), 자동화된 대화형 벤치마크 등
    - 여기서는 자동화된 대화형 벤치마크와 비슷한 방법 구현

#### AlpacaEval 방식 (다른 LLM을 사용해 fine-tuning된 모델 응답 평가 자동화)
* 공개 벤치마크 데이터셋 말고 예제의 테스트 세트 사용

#### generate 함수 -> test_data를 순회하여 testset 응답 생성하기
* test_data 딕셔너리에 생성된 모델 응답을 추가하여 instruction-data-with-response.json 파일에 저장

In [18]:
from tqdm import tqdm

for i, entry in tqdm(enumerate(test_data), total=len(test_data)):
    input_text = format_input(entry)

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )

    generated_text = token_ids_to_text(token_ids, tokenizer)

    response_text = (
        generated_text[len(input_text):]
        .replace("### Response:", "")
        .strip()
    )

    test_data[i]["model_response"] = response_text

with open("instruction-data-with-response.json", "w") as file:
    json.dump(test_data, file, indent=4)  # 읽기 쉽도록 들여쓰기 설정

  0%|          | 0/110 [00:00<?, ?it/s]


NameError: name 'generate' is not defined

In [ ]:
print(test_data[0])

#### 모델을 gpt2-medium355M-sft.pth 파일로 저장
* 저장된 모델은 model.load_state_dict(torch.load("gpt2-medium355M-sft.pth"))로 로드

In [ ]:
import re

file_name = f"{re.sub(r'[ ()]', '', CHOOSE_MODEL)}-sft.pth"  # 파일 이름에서 공백과 괄호 제거
torch.save(model.state_dict(), file_name)
print(f"모델이 {file_name}에 저장되었습니다.")

## 7.8. fine-tuning된 LLM 평가
* testset 응답 자동 평가를 위해 8B 파라미터 규모의 Llama3 모델의 instruction fine-tuning 된 버전 사용
* 오픈 소스 Ollama 애플리케이션(https://ollama.com)을 사용해 다운로드 가능


#### Ollama 사용
* 터미널에 ollama serve 명령

#### ollama run llama3 실행
    - 8B Llama 3 모델 다운로드 + 시작 (4.7 GB)


#### /bye로 ollam run llama3 세션 종료

#### Ollama 세션 실행 여부 확인

In [1]:
import psutil

def check_if_running(process_name):
    running = False

    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break

    return running


ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama가 실행 중이 아닙니다. 먼저 Ollama를 실행하세요."
    )

print("Ollama 실행:", check_if_running("ollama"))

Ollama 실행: True


#### 새로운 파이썬 세션에서 코드 실행

#### 모델과 상호작용하는 데 ollama run 명령 대신 REST API 사용 가능

In [3]:
import urllib.request
import json


def query_model(
    prompt,
    model="llama3",
    url="http://localhost:11434/api/chat"
):
    data = {
        # 페이로드(payload) 데이터를 딕셔너리로 만듦
        "model": model,
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "options": {
            # 결정론적인 응답을 위해 설정
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # 딕셔너리를 JSON 형식의 문자열로 변환하고 바이트로 인코딩
    payload = json.dumps(data).encode("utf-8")

    # method를 POST로 지정하고 필요한 헤더를 추가하여 요청 객체 생성
    request = urllib.request.Request(
        url,
        data=payload,
        method="POST"
    )

    request.add_header("Content-Type", "application/json")

    response_data = ""

    # 요청을 보내고 응답 받기
    with urllib.request.urlopen(request) as response:
        while True:
            line = response.readline().decode("utf-8")

            if not line:
                break

            response_json = json.loads(line)
            response_data += response_json["message"]["content"]

    return response_data

#### query_model 함수 사용 방법

In [4]:
model = "llama3"
result = query_model("What do Llamas eat?", model)

print(result)

Llamas are herbivores, which means they primarily feed on plant-based foods. Their diet typically consists of:

1. Grasses: Llamas love to graze on various types of grasses, including tall grasses, short grasses, and even weeds.
2. Hay: High-quality hay, such as alfalfa or timothy hay, is a staple in a llama's diet. They enjoy the sweet taste and texture of fresh hay.
3. Grains: Llamas may receive grains like oats, barley, or corn as part of their daily ration. However, it's essential to provide these grains in moderation, as they can be high in calories.
4. Fruits and vegetables: Llamas enjoy a variety of fruits and veggies, such as apples, carrots, sweet potatoes, and leafy greens like kale or spinach.
5. Minerals: Llamas require access to mineral supplements, which help maintain their overall health and well-being.

In the wild, llamas might also eat:

1. Leaves: They'll munch on leaves from trees and shrubs, including plants like willow, alder, and birch.
2. Bark: In some cases, ll

#### query_model 함수로 fine-tuning 된 모델이 생성한 응답 평가 가능
* Llama3 모델에게 참조로 제공된 텍스트 세트 응답을 바탕으로 모델의 응답을 0~100 사이의 척도로 평가하도록 요청

In [17]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}` "
        f"on a scale from 0 to 100, where 100 is the best score. "
    )

    print("\n데이터셋 응답:")
    print(">>", entry["output"])

    print("\n모델 응답:")
    print(">>", entry["model_response"])

    print("\n점수:")
    print(">>", query_model(prompt))

    print("\n-------------------------")

KeyError: 'model_response'

#### 점수만 반환하도록 프롬프트 수정 (100점 만점)

In [ ]:
from tqdm import tqdm


def generate_model_scores(json_data, json_key, model="llama3"):
    scores = []

    for entry in tqdm(json_data, desc="평가 항목"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}` "
            f"on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only "  # 점수만 반환하도록 추가된 명령
        )

        score = query_model(prompt, model)

        try:
            scores.append(int(score))
        except ValueError:
            print(f"점수로 변환할 수 없습니다: {score}")
            continue

    return scores

#### test_data에 generate_model_scores 함수 적용

In [ ]:
scores = generate_model_scores(test_data, "model_response")

print(f"평가 횟수: {len(test_data)}개 중 {len(scores)}개")

if scores:
    print(f"평균 점수: {sum(scores) / len(scores):.2f}\n")
else:
    print("평균 점수: 계산할 수 없음\n")

#### 모델 성능 향상을 위한 방법들
* fine-tuning 과정에서 학습률, batch_size, num_epochs 같은 하이퍼파라미터 조정
* train_set의 크기를 증가시키거나 다양한 주제/스타일을 포함하도록 샘플 다양화
* 모델 응답을 더 효과적으로 유도하기 위해 다른 프롬프트/instructino fomaat 실험
* 대규모 사전 훈련된 모델 사용

## 7.9. 결론

* 다음 단계
    - preference fine-tuning: 모델을 사람의 특정 선호도에 맞도록 커스터마이즈
    - 04_preference-tuning-with-dpo 폴더(https://bit.ly/4l12yux)

* LLM 관련 최신 정보
    - https://magazine.sebastianraschka.com/ (저자 웹사이트)
    - https://sebastianraschka.com/blog/ (저자 블로그)
    - https://arxiv.org/list/cs.LG/recent (아카이브 최신 연구 논문)

* 실전 애플리케이션을 위해 다양하고 강력한 LLM을 활용하고 싶다면
    - Axolotl (https://github.com/OpenAccess-AI-Collective/axolotl)
    - LitGPT (https://github.com/Lightning-AI/litgpt)
